# RVC — Hướng Dẫn Inference Chi Tiết
## Chuyển giọng nguồn → giọng đích từng bước

**Inference là gì?** Sau khi train xong model giọng đích, bước inference (infer) là chạy model đó để **chuyển một file audio sang giọng của model**.

## Tổng quan pipeline

```
[File audio nguồn (.wav/.mp3/...)]   [Model .pth]   [Index .index]
          ↓                               ↓                ↓
   FFmpeg resample                   Load model         Load FAISS
   → mono 16kHz float                    ↓                ↓
          ↓                              ↓                ↓
      Hubert encode               ┌──────────────────────┐
   → content vector (768d)        │  Retrieval (blending) │
          ↓                       └──────────────────────┘
      F0 extract                          ↓
   → pitch curve                   net_g vocoder
          ↓                              ↓
   F0 shift (+/- semitone)      [Audio giọng đích (.wav)]
          ↓
      → vào net_g
```

**File cần có để chạy:**
- File audio nguồn (giọng bạn muốn chuyển)
- `assets/weights/<tên>.pth` — model giọng đích (từ bước train)
- `assets/hubert/hubert_base.pt` — Hubert model
- `assets/rmvpe/rmvpe.pt` — RMVPE (nếu dùng `f0_method="rmvpe"`)
- `assets/indices/<tên>.index` — FAISS index (tùy chọn, giúp chất lượng tốt hơn)

## Yêu cầu hệ thống

| Yêu cầu | Chi tiết |
|---------|----------|
| Python env | Đã cài `requirements.txt` (PyTorch, fairseq, soundfile, librosa...) |
| FFmpeg | Trên PATH — dùng để đọc audio nguồn |
| GPU (khuyến nghị) | Infer CPU được nhưng chậm (~10x) |
| Working directory | Phải là thư mục `rvc_standalone` |

---
# BƯỚC 0: Kiểm Tra Môi Trường

Chạy toàn bộ phần này trước. Nếu có lỗi ở đây, không cần chạy tiếp.

### 0.1 — Kiểm tra Working Directory và cấu trúc thư mục

In [ ]:
import os
import sys
import pathlib

CWD = pathlib.Path.cwd().resolve()
print("Working directory:", CWD)

cac_thu_muc_can_co = [
    ("infer",            "Code inference pipeline"),
    ("configs",          "Config JSON cho từng sample rate"),
    ("assets/hubert",    "Hubert model weights"),
    ("assets/weights",   "Thư mục chứa model .pth"),
    ("assets/rmvpe",     "RMVPE model (nếu dùng rmvpe)"),
]

print()
thieu = []
for thu_muc, mo_ta in cac_thu_muc_can_co:
    ton_tai = (CWD / thu_muc).is_dir()
    icon = "✓" if ton_tai else "✗ THIẾU"
    print(f"  {icon}  {thu_muc:25s} — {mo_ta}")
    if not ton_tai:
        thieu.append(thu_muc)

if thieu:
    print("\n❌ Thiếu thư mục. Mở Jupyter từ thư mục rvc_standalone.")
else:
    print("\n✅ Cấu trúc thư mục OK.")

### 0.2 — Kiểm tra GPU

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        vram = props.total_memory / 1024**3
        print(f"  GPU {i}: {props.name}  |  VRAM: {vram:.1f} GB")
    print("✅ Sẽ infer trên GPU (nhanh).")
else:
    print("⚠️  Không có GPU — infer trên CPU (chậm hơn ~10x).")

### 0.3 — Kiểm tra file model weights

In [ ]:
import pathlib

CWD = pathlib.Path.cwd().resolve()

weights_can_co = [
    ("assets/hubert/hubert_base.pt",  "Bắt buộc — encode giọng nguồn"),
    ("assets/rmvpe/rmvpe.pt",         "Cần nếu dùng f0_method='rmvpe' (khuyến nghị)"),
]

print("Weights cần thiết:\n")
for duong_dan, mo_ta in weights_can_co:
    f = CWD / duong_dan
    if f.exists():
        size_mb = f.stat().st_size / 1024**2
        print(f"  ✓  {duong_dan}  ({size_mb:.0f} MB)")
    else:
        print(f"  ✗  {duong_dan} — THIẾU  ({mo_ta})")

print()

# Liệt kê model .pth có sẵn
weights_dir = CWD / "assets" / "weights"
pth_files = list(weights_dir.glob("*.pth")) if weights_dir.exists() else []

print(f"Model .pth trong assets/weights/ ({len(pth_files)} file):")
if pth_files:
    for f in sorted(pth_files):
        size_mb = f.stat().st_size / 1024**2
        print(f"  • {f.name}  ({size_mb:.0f} MB)")
else:
    print("  (trống — chưa train hoặc chưa xuất model)")

print()

# Liệt kê index .index có sẵn
indices_dir = CWD / "assets" / "indices"
index_files = list(indices_dir.glob("*.index")) if indices_dir.exists() else []
log_index_files = list((CWD / "logs").rglob("added_*.index")) if (CWD / "logs").exists() else []
tat_ca_index = index_files + log_index_files

print(f"File FAISS .index có sẵn ({len(tat_ca_index)} file):")
if tat_ca_index:
    for f in sorted(tat_ca_index):
        size_mb = f.stat().st_size / 1024**2
        rel = f.relative_to(CWD) if f.is_relative_to(CWD) else f
        print(f"  • {rel}  ({size_mb:.1f} MB)")
else:
    print("  (trống — tạo index ở Bước 8 khi train, hoặc infer không dùng index)")

### 0.4 — Kiểm tra FFmpeg

In [ ]:
import subprocess

try:
    result = subprocess.run(
        ["ffmpeg", "-version"],
        capture_output=True, text=True, timeout=5
    )
    dong_dau = result.stdout.splitlines()[0] if result.stdout else result.stderr.splitlines()[0]
    print("✅ FFmpeg OK:", dong_dau)
except FileNotFoundError:
    print("❌ Không tìm thấy FFmpeg trên PATH.")
    print("   Cài FFmpeg và thêm vào PATH để đọc được mọi định dạng audio.")
except Exception as e:
    print("⚠️  Lỗi kiểm tra FFmpeg:", e)

---
# BƯỚC 1: Chuẩn Bị File Audio Nguồn

**Audio nguồn** là file bạn muốn chuyển giọng — ví dụ: bài hát do người A hát, sau infer sẽ nghe như người B (giọng model) hát.

### Yêu cầu về audio nguồn

| Tiêu chí | Chi tiết |
|----------|----------|
| Định dạng | `.wav`, `.mp3`, `.flac`, `.ogg`, `.m4a`... (FFmpeg đọc được) |
| Nội dung | Giọng nói hoặc hát. Có nhạc nền được nhưng chất lượng sẽ kém hơn |
| Thời lượng | Không giới hạn. File dài → infer lâu hơn |
| Chất lượng tốt nhất | Chỉ có giọng, không có reverb, không có echo |

> **Mẹo:** Nếu audio nguồn có nhạc nền, hãy tách vocal trước bằng Demucs hoặc UVR5, rồi infer phần vocal, sau đó mix lại với nhạc nền.

In [ ]:
import pathlib
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# SỬA ĐÂY: Đường dẫn file audio nguồn
# Hỗ trợ: .wav .mp3 .flac .ogg .m4a (bất kỳ định dạng FFmpeg đọc được)
# ============================================================
INPUT_WAV = r"C:\đường\dẫn\đến\audio\nguon.wav"  # <-- SỬA ĐƯỜNG DẪN NÀY

# Đường dẫn file output (kết quả sau infer)
OUTPUT_WAV = "infer_output.wav"  # Có thể để mặc định hoặc đổi tên

# Kiểm tra file tồn tại
input_path = pathlib.Path(INPUT_WAV)
if not input_path.exists():
    print(f"❌ File không tồn tại: {INPUT_WAV}")
    print("   Sửa biến INPUT_WAV ở trên cho đúng đường dẫn.")
else:
    size_mb = input_path.stat().st_size / 1024**2
    print(f"✅ File audio nguồn: {input_path.name}  ({size_mb:.1f} MB)")

    # Xem thời lượng bằng librosa
    try:
        import librosa
        y, sr = librosa.load(str(input_path), sr=None, mono=False)
        if y.ndim == 1:
            kenh, mau = 1, len(y)
        else:
            kenh, mau = y.shape[0], y.shape[1]
        thoi_luong = mau / sr
        phut = int(thoi_luong // 60)
        giay = thoi_luong % 60
        print(f"   Sample rate : {sr:,} Hz")
        print(f"   Kênh        : {kenh} ({'mono' if kenh==1 else 'stereo'})")
        print(f"   Thời lượng  : {phut}:{giay:05.2f} phút")
    except Exception as e:
        print(f"   (Không đọc được thông tin chi tiết: {e})")

### 1.1 — Nghe thử audio nguồn (tùy chọn)

In [ ]:
# Phát audio nguồn ngay trong notebook
import pathlib

try:
    from IPython.display import Audio, display
    if pathlib.Path(INPUT_WAV).exists():
        print("Audio nguồn (giọng sẽ được chuyển):")
        display(Audio(INPUT_WAV))
    else:
        print("File chưa tồn tại — sửa INPUT_WAV trước.")
except ImportError:
    print("IPython không có — bỏ qua preview.")

---
# BƯỚC 2: Khởi Tạo Môi Trường Python

Chạy **một lần** sau khi mở notebook hoặc sau Restart kernel.

In [ ]:
import os
import sys
import pathlib

# Setup working directory
STANDALONE_ROOT = pathlib.Path.cwd().resolve()
if not (STANDALONE_ROOT / "infer" / "modules" / "vc" / "modules.py").is_file():
    raise SystemExit(
        "\n❌ Working directory sai!"
        "\n   Phải chạy từ thư mục rvc_standalone."
    )

os.chdir(STANDALONE_ROOT)
if str(STANDALONE_ROOT) not in sys.path:
    sys.path.insert(0, str(STANDALONE_ROOT))

# Đặt các đường dẫn mặc định (giống .env)
os.environ.setdefault("weight_root",        "assets/weights")
os.environ.setdefault("index_root",          "logs")
os.environ.setdefault("outside_index_root",  "assets/indices")
os.environ.setdefault("rmvpe_root",          "assets/rmvpe")

# Load .env nếu có
try:
    from dotenv import load_dotenv
    load_dotenv(os.path.join(STANDALONE_ROOT, ".env"), override=False)
except ImportError:
    pass

print("✅ Môi trường sẵn sàng.")
print("   Root:", STANDALONE_ROOT)
print("   weight_root:", os.environ["weight_root"])
print("   index_root :", os.environ["index_root"])
print("   rmvpe_root :", os.environ["rmvpe_root"])

---
# BƯỚC 3: Tải Config (GPU / Device / FP16)

`Config` tự phát hiện GPU, đặt `device` (cuda:0 hoặc cpu) và `is_half` (FP16). Cũng tính các thông số chunk inference (`x_pad`, `x_query`, `x_center`, `x_max`) để không bị OOM VRAM.

| Biến Config | Ý nghĩa |
|-------------|----------|
| `config.device` | `"cuda:0"` hoặc `"cpu"` — thiết bị tính toán |
| `config.is_half` | `True` nếu GPU hỗ trợ FP16 — giảm VRAM, tăng tốc |
| `config.x_pad` | Padding hai đầu chunk (giây) |
| `config.x_query` | Kích thước query chunk (giây) |
| `config.x_center` | Phần trung tâm chunk lấy làm output |
| `config.x_max` | Độ dài tối đa mỗi chunk xử lý |

In [ ]:
import sys

# Tạm reset sys.argv để tránh Jupyter truyền tham số lạ vào argparse của Config
_saved_argv = sys.argv
sys.argv = ["rvc_infer"]

try:
    from configs.config import Config
    config = Config()
finally:
    sys.argv = _saved_argv

print("=" * 45)
print("CONFIG")
print("=" * 45)
print(f"  device      : {config.device}")
print(f"  is_half     : {config.is_half}   (FP16)")
print(f"  x_pad       : {config.x_pad}    giây (padding chunk)")
print(f"  x_query     : {config.x_query}    giây (kích thước query)")
print(f"  x_center    : {config.x_center}   giây (kích thước center)")
print(f"  x_max       : {config.x_max}   giây (tối đa mỗi chunk)")
print("=" * 45)
print("✅ Config OK.")

---
# BƯỚC 4: Chọn và Tải Model Giọng Đích

**Model giọng đích** là file `.pth` được tạo ra từ bước train. Nó chứa:
- Kiến trúc mạng (v1/v2, có/không F0)
- Trọng số đã học (timbre của giọng đích)
- Metadata: sample rate, version, speaker count

**Nơi lưu:** `assets/weights/<tên>.pth`

In [ ]:
import pathlib

# Liệt kê lại các model có sẵn để dễ chọn
weights_dir = pathlib.Path("assets") / "weights"
pth_files = sorted(weights_dir.glob("*.pth")) if weights_dir.exists() else []

if pth_files:
    print("Các model .pth có sẵn:\n")
    for i, f in enumerate(pth_files):
        size_mb = f.stat().st_size / 1024**2
        print(f"  [{i}] {f.name}  ({size_mb:.0f} MB)")
else:
    print("❌ Không có file .pth trong assets/weights/")
    print("   Chạy Bước 9 trong notebook training để xuất model.")

In [ ]:
# ============================================================
# SỬA ĐÂY: Tên file model .pth (chỉ tên file, không cần đường dẫn đầy đủ)
# Model phải nằm trong assets/weights/
# ============================================================
MODEL_FILE = "giong_A_infer.pth"  # <-- SỬA TÊN NÀY

# Kiểm tra file tồn tại
model_path = pathlib.Path("assets") / "weights" / MODEL_FILE
if model_path.exists():
    size_mb = model_path.stat().st_size / 1024**2
    print(f"✅ Model: {MODEL_FILE}  ({size_mb:.0f} MB)")
else:
    print(f"❌ Không tìm thấy: {model_path}")
    print("   Kiểm tra lại tên file ở trên.")

In [ ]:
from infer.modules.vc.modules import VC

# Tạo đối tượng VC và nạp model
vc = VC(config)
_ = vc.get_vc(MODEL_FILE)  # Load checkpoint vào vc.net_g

if vc.net_g is None:
    print("❌ Load model thất bại. Kiểm tra tên file và weight_root.")
else:
    print("=" * 45)
    print("THÔNG TIN MODEL")
    print("=" * 45)
    print(f"  Model file   : {MODEL_FILE}")
    print(f"  Sample rate  : {vc.tgt_sr} Hz  (output sẽ ở sample rate này)")
    print(f"  Version      : {vc.version}")
    print(f"  Có F0 (pitch): {bool(vc.if_f0)}")
    try:
        n_spk = vc.cpt["config"][-3]
        print(f"  Speaker count: {n_spk}  (speaker_id từ 0 đến {n_spk-1})")
    except Exception:
        pass
    print("=" * 45)
    print("✅ Model đã tải.")

---
# BƯỚC 5: Tải Hubert Model

Hubert encode audio nguồn thành **content vector** — biểu diễn nội dung âm thanh (phoneme, âm tiết) không phụ thuộc vào timbre. Sau đó net_g mới "thay" timbre thành giọng đích.

File: `assets/hubert/hubert_base.pt` (~360 MB). Chỉ cần tải một lần, tái dùng cho nhiều lần infer.

In [ ]:
from infer.modules.vc.utils import load_hubert

print("Đang tải Hubert model...")
vc.hubert_model = load_hubert(config)

device_hubert = next(vc.hubert_model.parameters()).device
print(f"✅ Hubert đã tải — đang chạy trên: {device_hubert}")

---
# BƯỚC 6: Cấu Hình FAISS Index (Tùy Chọn)

FAISS Index giúp infer **tự nhiên hơn** bằng cách tìm kiếm vector feature gần nhất trong tập training rồi blend vào. Nếu không có index, infer vẫn chạy được.

**Khi nào nên dùng index?**
- Khi muốn giọng infer giống giọng đích nhất có thể
- Khi index được tạo từ cùng model đang dùng

**Khi nào bỏ qua (INDEX_PATH = "")?**
- Khi chưa tạo index
- Khi muốn tốc độ nhanh hơn
- Khi đặt `INDEX_RATE = 0` (index không có tác dụng dù có file)

In [ ]:
import pathlib

# Liệt kê các file index có sẵn
cac_index = []
for thu_muc in ["assets/indices", "logs"]:
    d = pathlib.Path(thu_muc)
    if d.exists():
        cac_index += list(d.rglob("added_*.index"))
        cac_index += list(d.glob("*.index"))

# Loại trùng
cac_index = list({str(f): f for f in cac_index}.values())

if cac_index:
    print(f"Các file .index có sẵn ({len(cac_index)} file):\n")
    for f in sorted(cac_index):
        size_mb = f.stat().st_size / 1024**2
        print(f"  • {f}  ({size_mb:.1f} MB)")
else:
    print("Không có file .index — infer sẽ chạy không dùng retrieval.")

In [ ]:
# ============================================================
# SỬA ĐÂY: Đường dẫn file .index
# Để rỗng ("") nếu không muốn dùng index
# ============================================================
INDEX_PATH = ""  # <-- SỬA: ví dụ r".\assets\indices\giong_A_added_IVF326_Flat_nprobe_1_giong_A_v2.index"

if not INDEX_PATH:
    print("⚠️  Không dùng FAISS index.")
    print("   Infer vẫn chạy được. Đặt INDEX_RATE = 0 hoặc để INDEX_PATH rỗng.")
else:
    idx_path = pathlib.Path(INDEX_PATH)
    if idx_path.exists():
        size_mb = idx_path.stat().st_size / 1024**2
        print(f"✅ Index: {idx_path.name}  ({size_mb:.1f} MB)")
    else:
        print(f"❌ Không tìm thấy file index: {INDEX_PATH}")

---
# BƯỚC 7: Cấu Hình Tất Cả Tham Số Inference

**Đây là phần quan trọng nhất — đọc kỹ bảng giải thích trước khi chỉnh.**

### Giải thích chi tiết từng tham số

---

#### SPEAKER_ID — Chỉ số Speaker

- **Kiểu:** `int`
- **Giá trị thường dùng:** `0`
- **Giải thích:** RVC hỗ trợ train nhiều giọng trong 1 model (multi-speaker). Nếu train 1 giọng (thường gặp), luôn để `0`. Nếu model có nhiều giọng, chọn ID từ 0 đến `n_spk-1` (xem Bước 4).

---

#### F0_UP_KEY — Dịch Tông (Pitch Shift)

- **Kiểu:** `int` (số nguyên, có thể âm)
- **Đơn vị:** nửa cung (semitone)
- **Giá trị:** `-12` đến `+12` thường dùng
- **Giải thích:**
  - `0` = giữ nguyên tông gốc
  - `+12` = tăng 1 quãng tám (giọng cao lên)
  - `-12` = giảm 1 quãng tám (giọng thấp xuống)
  - `+5` hoặc `+6` nếu chuyển từ giọng nam → giọng nữ
  - `-5` hoặc `-6` nếu chuyển từ giọng nữ → giọng nam

| Trường hợp | Giá trị gợi ý |
|------------|---------------|
| Giữ nguyên tông | `0` |
| Nam → Nữ | `+5` đến `+7` |
| Nữ → Nam | `-5` đến `-7` |
| Cùng giới tính | `0` hoặc `±1-3` |

---

#### F0_METHOD — Phương Pháp Phân Tích Cao Độ

- **Kiểu:** `str`
- **Các giá trị:** `"pm"` `"harvest"` `"crepe"` `"rmvpe"`

| Phương pháp | Tốc độ | Chất lượng | Ghi chú |
|-------------|--------|-----------|--------|
| `pm` | ★★★★★ Nhanh nhất | ★★★ Tạm | Không cần file phụ |
| `harvest` | ★★ Chậm | ★★★★ Tốt | Ổn định với giọng thấp |
| `crepe` | ★★ Chậm | ★★★★ Tốt | Cần GPU, nặng VRAM |
| `rmvpe` | ★★★★ Nhanh | ★★★★★ Tốt nhất | **Khuyến nghị**, cần `rmvpe.pt` |

---

#### INDEX_RATE — Tỉ Lệ Trộn Index

- **Kiểu:** `float` từ `0.0` đến `1.0`
- **Công thức:** `features = index_rate × features_từ_index + (1 - index_rate) × features_từ_Hubert`
- **Giải thích:**
  - `0.0` = không dùng index (dù INDEX_PATH có file)
  - `0.75` = mặc định khuyến nghị
  - `1.0` = hoàn toàn dùng index (có thể artifact)
- **Lưu ý:** Chỉ có tác dụng nếu INDEX_PATH trỏ đến file tồn tại

---

#### FILTER_RADIUS — Lọc F0

- **Kiểu:** `int` từ `0` đến `7`
- **Giải thích:** Chỉ ảnh hưởng khi `F0_METHOD = "harvest"`. Áp dụng median filter lên đường cong F0 để giảm nhiễu pitch đột ngột.
  - `0` = tắt filter
  - `3` = mặc định, cân bằng
  - `7` = làm mượt mạnh (có thể mất chi tiết)
- **Với các method khác (rmvpe, pm, crepe):** Giá trị này không có tác dụng

---

#### RESAMPLE_SR — Resample Output

- **Kiểu:** `int` (Hz)
- **Giải thích:**
  - `0` = giữ nguyên sample rate của model (thường 40000 Hz với 40k model)
  - `44100` = resample output về 44.1kHz
  - `48000` = resample output về 48kHz
- **Khi nào dùng?** Khi bạn cần output ở sample rate cụ thể để ghép với nhạc nền (ví dụ nhạc nền 44.1kHz)

---

#### RMS_MIX_RATE — Trộn Biên Độ (Loudness)

- **Kiểu:** `float` từ `0.0` đến `1.0`
- **Giải thích:** Điều chỉnh envelope âm lượng của output:
  - `0.0` = giữ nguyên envelope của giọng đích (output thuần model)
  - `1.0` = copy hoàn toàn envelope của audio nguồn vào output
  - `0.25` = blend nhẹ, nghe tự nhiên hơn trong hầu hết trường hợp
- **Công thức:** `output = output × ((rms_input / rms_output) ^ rms_mix_rate)`

---

#### PROTECT — Bảo Vệ Phụ Âm

- **Kiểu:** `float` từ `0.0` đến `0.5`
- **Giải thích:** Bảo vệ các vùng âm vô thanh (phụ âm, "s", "t", "p"...) khỏi bị model xử lý sai:
  - `0.33` = mặc định, cân bằng tốt
  - `0.0` = bảo vệ tối đa (giữ nguyên phụ âm gốc)
  - `0.5` = tắt cơ chế bảo vệ
- **Khi nào điều chỉnh?** Nếu phụ âm nghe bị vỡ/artifact, giảm giá trị này xuống

In [ ]:
# ================================================================
# PHẦN CẦN SỬA — Đặt tất cả tham số inference
# ================================================================

# --- Speaker ID ---
# Thường là 0 nếu model được train 1 giọng
SPEAKER_ID = 0

# --- Pitch shift ---
# 0 = giữ nguyên, +12 = lên 1 quãng tám, -12 = xuống 1 quãng tám
# Nam → Nữ: thử +5 đến +7
# Nữ → Nam: thử -5 đến -7
F0_UP_KEY = 0

# --- Phương pháp phân tích F0 ---
# Khuyến nghị: "rmvpe" (cần rmvpe.pt)
# Nếu không có rmvpe.pt, dùng "pm" (nhanh) hoặc "harvest" (chất lượng hơn)
F0_METHOD = "rmvpe"

# --- Tỉ lệ blending với FAISS index ---
# 0.0 = không dùng index, 0.75 = mặc định, 1.0 = hoàn toàn dùng index
# Chỉ có tác dụng nếu INDEX_PATH không rỗng
INDEX_RATE = 0.75

# --- Lọc F0 (chỉ dùng với harvest) ---
# 0-7; 3 = mặc định; 0 = tắt
FILTER_RADIUS = 3

# --- Resample output ---
# 0 = giữ nguyên sample rate model (thường 40000)
# 44100 hoặc 48000 nếu cần output ở sample rate cụ thể
RESAMPLE_SR = 0

# --- Trộn biên độ âm thanh ---
# 0.0 = output thuần model, 0.25 = mặc định, 1.0 = copy envelope nguồn
RMS_MIX_RATE = 0.25

# --- Bảo vệ phụ âm ---
# 0.33 = mặc định, 0.0 = bảo vệ tối đa, 0.5 = tắt
PROTECT = 0.33

# --- File F0 ngoài (hầu như không dùng) ---
# None = tự phân tích F0 từ audio
# Đường dẫn file = dùng F0 từ file ngoài thay vì phân tích
F0_FILE = None

# ================================================================

print("Tham số đã đặt:")
print(f"  SPEAKER_ID   = {SPEAKER_ID}")
print(f"  F0_UP_KEY    = {F0_UP_KEY}  semitone{'s' if abs(F0_UP_KEY) != 1 else ''}")
print(f"  F0_METHOD    = '{F0_METHOD}'")
print(f"  INDEX_RATE   = {INDEX_RATE}  {'(không dùng index)' if not INDEX_PATH or INDEX_RATE == 0 else ''}")
print(f"  FILTER_RADIUS= {FILTER_RADIUS}  {'(chỉ dùng với harvest)' if F0_METHOD != 'harvest' else ''}")
print(f"  RESAMPLE_SR  = {RESAMPLE_SR}  {'Hz' if RESAMPLE_SR else '(giữ sr model)'}")
print(f"  RMS_MIX_RATE = {RMS_MIX_RATE}")
print(f"  PROTECT      = {PROTECT}")

---
# BƯỚC 8: Chạy Inference

Tất cả đã sẵn sàng. Ô này chạy toàn bộ pipeline và xuất file WAV.

In [ ]:
import soundfile as sf
import time

file_index  = INDEX_PATH.strip() if INDEX_PATH else ""
file_index2 = ""  # dropdown thứ hai trong WebUI, không cần ở notebook

print("Bắt đầu inference...")
print(f"  Input  : {INPUT_WAV}")
print(f"  Model  : {MODEL_FILE}")
print(f"  Index  : {file_index if file_index else '(không dùng)'}")
print(f"  Output : {OUTPUT_WAV}")
print()

bat_dau = time.time()

info, audio_out = vc.vc_single(
    SPEAKER_ID,
    INPUT_WAV,
    F0_UP_KEY,
    F0_FILE,
    F0_METHOD,
    file_index,
    file_index2,
    INDEX_RATE,
    FILTER_RADIUS,
    RESAMPLE_SR,
    RMS_MIX_RATE,
    PROTECT,
)

thoi_gian = time.time() - bat_dau

print("Log từ pipeline:")
print(info)
print()

if audio_out is None:
    print("❌ Inference thất bại. Xem log ở trên.")
else:
    tgt_sr, audio_i16 = audio_out
    sf.write(OUTPUT_WAV, audio_i16, tgt_sr)

    thoi_luong_out = len(audio_i16) / tgt_sr
    print(f"✅ Inference xong trong {thoi_gian:.1f}s")
    print(f"   Output: {OUTPUT_WAV}")
    print(f"   Sample rate: {tgt_sr} Hz")
    print(f"   Thời lượng : {int(thoi_luong_out//60)}:{thoi_luong_out%60:05.2f} phút")
    print(f"   Tốc độ     : {thoi_luong_out/thoi_gian:.1f}x realtime")

---
# BƯỚC 9: Nghe Kết Quả và So Sánh

In [ ]:
import pathlib
from IPython.display import Audio, display

print("=" * 50)
print("SO SÁNH AUDIO")
print("=" * 50)

print("\n🎤 Audio NGUỒN (giọng ban đầu):")
display(Audio(INPUT_WAV))

if pathlib.Path(OUTPUT_WAV).exists():
    print(f"\n🎵 Audio OUTPUT (đã chuyển sang giọng '{MODEL_FILE}'):")
    display(Audio(OUTPUT_WAV))
else:
    print("\n❌ File output chưa có — chạy Bước 8 trước.")

---
# BƯỚC 10: Infer Nhiều File (Batch)

Nếu có nhiều file audio cần chuyển giọng, dùng ô này thay vì chạy từng file một.

In [ ]:
import pathlib
import soundfile as sf
import time

# ============================================================
# SỬA ĐÂY cho Batch Inference
# ============================================================

# Thư mục chứa các file audio nguồn
BATCH_INPUT_DIR  = r"C:\đường\dẫn\thư_mục_input"   # <-- SỬA

# Thư mục lưu kết quả output
BATCH_OUTPUT_DIR = r"C:\đường\dẫn\thư_mục_output"  # <-- SỬA

# Định dạng file cần xử lý
BATCH_EXTENSIONS = [".wav", ".mp3", ".flac", ".ogg", ".m4a"]

# Tham số infer (dùng lại từ Bước 7 hoặc ghi đè ở đây)
batch_params = dict(
    f0_up_key     = F0_UP_KEY,
    f0_file       = F0_FILE,
    f0_method     = F0_METHOD,
    file_index    = INDEX_PATH.strip() if INDEX_PATH else "",
    file_index2   = "",
    index_rate    = INDEX_RATE,
    filter_radius = FILTER_RADIUS,
    resample_sr   = RESAMPLE_SR,
    rms_mix_rate  = RMS_MIX_RATE,
    protect       = PROTECT,
)

# ============================================================

input_dir  = pathlib.Path(BATCH_INPUT_DIR)
output_dir = pathlib.Path(BATCH_OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

if not input_dir.exists():
    print(f"❌ Thư mục input không tồn tại: {input_dir}")
else:
    files = [f for f in sorted(input_dir.iterdir())
             if f.suffix.lower() in BATCH_EXTENSIONS]

    print(f"Tìm thấy {len(files)} file audio trong {input_dir}\n")

    for i, audio_file in enumerate(files, 1):
        output_file = output_dir / (audio_file.stem + "_converted.wav")
        print(f"[{i}/{len(files)}] {audio_file.name}")

        bat_dau = time.time()
        try:
            info, audio_out = vc.vc_single(
                SPEAKER_ID,
                str(audio_file),
                **batch_params,
            )
            if audio_out is None:
                print(f"  ❌ Lỗi: {info}")
            else:
                tgt_sr, audio_i16 = audio_out
                sf.write(str(output_file), audio_i16, tgt_sr)
                elapsed = time.time() - bat_dau
                print(f"  ✅ Xong ({elapsed:.1f}s) → {output_file.name}")
        except Exception as e:
            print(f"  ❌ Exception: {e}")

    print(f"\n🎉 Batch xong. Kết quả trong: {output_dir}")

---
# BƯỚC 11: Hướng Dẫn Tinh Chỉnh Tham Số

Nếu kết quả không như ý, thử điều chỉnh theo bảng sau:

## Vấn đề thường gặp và cách khắc phục

| Vấn đề | Nguyên nhân có thể | Cách khắc phục |
|--------|-------------------|-----------------|
| Giọng sai tông (quá cao/thấp) | F0_UP_KEY chưa đúng | Điều chỉnh `F0_UP_KEY` ±1 cho đến khi đúng tông |
| Âm thanh nghe "robotic" | Model train ít epoch | Train thêm epoch, hoặc thêm data |
| Phụ âm bị vỡ/artifact | PROTECT quá cao | Giảm `PROTECT` về 0.1-0.2 |
| Giọng không giống model | INDEX_RATE quá thấp | Tăng `INDEX_RATE` lên 0.75-0.9 |
| Âm thanh bị vỡ/artifact nhiều | INDEX_RATE quá cao | Giảm `INDEX_RATE` xuống 0.3-0.5 |
| Âm lượng output khác input | RMS_MIX_RATE chưa đúng | Tăng `RMS_MIX_RATE` về 0.5-0.75 |
| F0 nhảy pitch bất thường | F0_METHOD không phù hợp | Đổi sang `rmvpe` hoặc `harvest` |
| Lỗi OOM (Out of Memory) | File audio quá dài hoặc VRAM thiếu | Chia file audio thành đoạn ngắn hơn |
| Lỗi `path does not exist` | INPUT_WAV sai đường dẫn | Kiểm tra lại đường dẫn file |
| Lỗi `RMVPE` không load | Thiếu `assets/rmvpe/rmvpe.pt` | Chuyển sang `F0_METHOD = "pm"` hoặc tải file về |

## Gợi ý tham số theo từng use case

| Use case | F0_UP_KEY | F0_METHOD | INDEX_RATE | PROTECT |
|----------|-----------|-----------|------------|----------|
| Hát (cùng giới tính) | 0 | rmvpe | 0.75 | 0.33 |
| Hát nam → nữ | +5 đến +7 | rmvpe | 0.75 | 0.33 |
| Hát nữ → nam | -5 đến -7 | rmvpe | 0.75 | 0.33 |
| Giọng nói (podcast) | 0 | pm | 0.5 | 0.3 |
| Giọng nói, cần tốc độ | 0 | pm | 0 | 0.5 |
| Chất lượng tối đa | 0 | rmvpe | 0.85 | 0.25 |

---
# BƯỚC 12: Infer Nhanh (Chạy Lại Sau Khi Đã Setup)

Sau khi đã chạy qua một lần đầy đủ (Bước 0-9), lần sau chỉ cần chạy các ô này:

> **Lưu ý:** Bước 2, 3, 4, 5 chỉ cần chạy 1 lần sau khi mở notebook hoặc restart kernel.

In [ ]:
# ============================================================
# QUICK INFER — Sửa 3 biến này và chạy ô này
# ============================================================

INPUT_WAV_QUICK  = r"C:\đường\dẫn\audio_nguon.wav"   # File audio nguồn
OUTPUT_WAV_QUICK = "output_quick.wav"                 # File output
F0_UP_KEY_QUICK  = 0                                  # Pitch shift

# ============================================================

import soundfile as sf
import time

bat_dau = time.time()

info, audio_out = vc.vc_single(
    SPEAKER_ID,
    INPUT_WAV_QUICK,
    F0_UP_KEY_QUICK,
    F0_FILE,
    F0_METHOD,
    INDEX_PATH.strip() if INDEX_PATH else "",
    "",
    INDEX_RATE,
    FILTER_RADIUS,
    RESAMPLE_SR,
    RMS_MIX_RATE,
    PROTECT,
)

if audio_out:
    tgt_sr, audio_i16 = audio_out
    sf.write(OUTPUT_WAV_QUICK, audio_i16, tgt_sr)
    elapsed = time.time() - bat_dau
    print(f"✅ Xong ({elapsed:.1f}s) → {OUTPUT_WAV_QUICK}")
    from IPython.display import Audio, display
    display(Audio(OUTPUT_WAV_QUICK))
else:
    print("❌", info)

---
# Tổng Kết — Checklist Trước Khi Infer

```
☐ Working directory = rvc_standalone
☐ assets/hubert/hubert_base.pt  tồn tại
☐ assets/rmvpe/rmvpe.pt         tồn tại  (nếu dùng rmvpe)
☐ assets/weights/<model>.pth    tồn tại
☐ assets/indices/<index>.index  tồn tại  (tùy chọn)
☐ File audio nguồn tồn tại
☐ F0_UP_KEY phù hợp với giọng (nam/nữ)
☐ SPEAKER_ID = 0 (nếu model 1 giọng)
```

## Các file cần có để chạy notebook này

| File / Thư mục | Bắt buộc | Kích thước |
|----------------|----------|-----------|
| `infer/` | ✅ | — |
| `configs/` | ✅ | — |
| `assets/hubert/hubert_base.pt` | ✅ | ~360 MB |
| `assets/rmvpe/rmvpe.pt` | ⚠️ (nếu dùng rmvpe) | ~180 MB |
| `assets/weights/<tên>.pth` | ✅ | ~60 MB |
| `assets/indices/<tên>.index` | Tùy chọn | Nhỏ |
| File audio nguồn | ✅ | Tùy |
| `requirements.txt` đã cài | ✅ | — |